###  Libraries required: 

In [5780]:
import pandas as pd
import math
import numpy as np
from datetime import datetime
import requests
import re
from scipy.stats import t
    

### Utils:

In [5783]:
def t_upper_bound(mean, std, n, confidence=0.95):
    df = n - 1
    t_crit = t.ppf((1 + confidence) / 2, df)
    se = std / np.sqrt(n)
    return mean + t_crit * se

def z_upper_bound(mean, std, n):
    return mean + (1.96 * (std/np.sqrt(n)))

def preprocessing_names(x):
    pattern = "[^a-zA-Z0-9 ]"
    val = re.sub(pattern, "", x).lower()
    val = val.split()
    val = ' '.join(val)
    return val

def q(x, arr):
    n = len(x)
    qs = {}
    for i in  arr:
        threshold = math.ceil(n*i)
        qs[f"{round(i*100)}%"] = [x[threshold], n - np.ceil(n*i), np.ceil(n*i)]
    return qs

def highlight_rows(row, colour, column_name, column_values):
    if row[f'{column_name}'] in f'{column_values}':
        return [f'background-color: {colour}'] * len(row)
    else:
        return [''] * len(row)

colours = ['yellow','lightyellow','gold','orange','lightcoral','red','pink','lightpink','salmon','lightgreen',
          'green','lime','palegreen','lightblue','skyblue','deepskyblue','blue','lightcyan','turquoise',
          'purple','violet','plum','lavender','gray','lightgray','silver','white','black'
]

## Data Preprocessing and Exploration

### Loading the data

In [5788]:
df = pd.read_csv("Nassau Candy Distributor.csv")
df_factories = pd.read_csv("factories.csv")
df_products = pd.read_csv("products_factories.csv")

In [5790]:
df.head(2)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.5,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.5,2,4.90,2.60


### Exploration

In [5793]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Country/Region', 'City', 'State/Province',
       'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name',
       'Sales', 'Units', 'Gross Profit', 'Cost'],
      dtype='object')

In [5795]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

In [5797]:
df.describe()

,Row ID,Customer ID,Sales,Units,Gross Profit,Cost
count,10194.000000,10194.000000,10194.000000,10194.000000,10194.000000,10194.000000
mean,5097.500000,134468.961154,13.908537,3.791838,9.166451,4.742087
std,2942.898656,20231.483007,11.341020,2.228317,6.643740,5.061647
min,1.000000,100006.000000,1.250000,1.000000,0.250000,0.600000
25%,2549.250000,117212.000000,7.200000,2.000000,4.900000,2.400000
50%,5097.500000,133550.000000,10.800000,3.000000,7.470000,3.600000
75%,7645.750000,152051.000000,18.000000,5.000000,12.250000,5.700000
max,10194.000000,192314.000000,260.000000,14.000000,130.000000,130.000000


In [5799]:
df.isna().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
dtype: int64

### Note on Duplicated `order_id`s

Duplicated `order_id`s are present in the data, possibly due to:

1. A customer ordering different products.
2. Redundant data entries.
3. Unique information where `order_id` is repeated unnecessarily.

In [5802]:
df[df['Order ID'].duplicated()].reset_index().sort_values('Order ID')

,index,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
15,64,65,CA-2021-115238-CHO-FUD-51000,21-01-2024,17-07-2026,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-FUD-51000,Wonka Bar - Fudge Mallows,3.60,1,2.40,1.20
16,65,66,CA-2021-115238-CHO-MIL-31000,21-01-2024,17-07-2026,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,32.50,10,21.10,11.40
328,2022,2023,CA-2021-123625-CHO-MIL-31000,29-12-2024,25-06-2027,Standard Class,123625,Canada,Quebec City,Quebec,G1B,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,16.25,5,10.55,5.70
325,2019,2020,CA-2021-123625-CHO-MIL-31000,29-12-2024,25-06-2027,Standard Class,123625,Canada,Quebec City,Quebec,G1B,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
326,2020,2021,CA-2021-123625-CHO-MIL-31000,29-12-2024,25-06-2027,Standard Class,123625,Canada,Quebec City,Quebec,G1B,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,8068,8069,US-2024-169551-CHO-TRI-54000,07-07-2025,30-12-2029,First Class,169551,United States,Philadelphia,Pennsylvania,19120,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
1294,8066,8067,US-2024-169551-CHO-TRI-54000,07-07-2025,30-12-2029,First Class,169551,United States,Philadelphia,Pennsylvania,19120,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
1606,9975,9976,US-2024-169859-CHO-MIL-31000,14-12-2025,10-06-2030,Standard Class,169859,United States,San Diego,California,92024,Chocolate,Pacific,CHO-MIL-31000,Wonka Bar - Milk Chocolate,16.25,5,10.55,5.70
1544,9635,9636,US-2024-169894-CHO-TRI-54000,24-11-2025,22-05-2030,Second Class,169894,United States,Mesa,Arizona,85204,Chocolate,Pacific,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,30.00,8,19.60,10.40


In [5804]:
df[df['Order ID'].duplicated()].reset_index().sort_values('Order ID')

,index,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
15,64,65,CA-2021-115238-CHO-FUD-51000,21-01-2024,17-07-2026,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-FUD-51000,Wonka Bar - Fudge Mallows,3.60,1,2.40,1.20
16,65,66,CA-2021-115238-CHO-MIL-31000,21-01-2024,17-07-2026,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,32.50,10,21.10,11.40
328,2022,2023,CA-2021-123625-CHO-MIL-31000,29-12-2024,25-06-2027,Standard Class,123625,Canada,Quebec City,Quebec,G1B,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,16.25,5,10.55,5.70
325,2019,2020,CA-2021-123625-CHO-MIL-31000,29-12-2024,25-06-2027,Standard Class,123625,Canada,Quebec City,Quebec,G1B,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
326,2020,2021,CA-2021-123625-CHO-MIL-31000,29-12-2024,25-06-2027,Standard Class,123625,Canada,Quebec City,Quebec,G1B,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,8068,8069,US-2024-169551-CHO-TRI-54000,07-07-2025,30-12-2029,First Class,169551,United States,Philadelphia,Pennsylvania,19120,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
1294,8066,8067,US-2024-169551-CHO-TRI-54000,07-07-2025,30-12-2029,First Class,169551,United States,Philadelphia,Pennsylvania,19120,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
1606,9975,9976,US-2024-169859-CHO-MIL-31000,14-12-2025,10-06-2030,Standard Class,169859,United States,San Diego,California,92024,Chocolate,Pacific,CHO-MIL-31000,Wonka Bar - Milk Chocolate,16.25,5,10.55,5.70
1544,9635,9636,US-2024-169894-CHO-TRI-54000,24-11-2025,22-05-2030,Second Class,169894,United States,Mesa,Arizona,85204,Chocolate,Pacific,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,30.00,8,19.60,10.40


In [5806]:
len(df['Order ID'].unique())

8549

In [5808]:
df.duplicated().sum()

0

### Dropping the `Row ID` column due to:
1. Serves no purpose
2. Uniquely identifies duplicated information

In [5811]:
df.drop(columns=['Row ID'], inplace=True)

### Dropping the Row ID column has made an impact:
- Identified the duplicated data 

In [5814]:
df.duplicated().sum()

313

In [5816]:
df.drop_duplicates(inplace=True)

### Still `1332 Order IDs` are duplicated

In [5819]:
df['Order ID'].duplicated().sum()

1332

### Redundancy in the `Order IDs` can be seen due to the variation in the `Sales`, `Units`, `Gross` `Profit` and `Cost`.
1. Same products being bought by the same customer, on same date, but different quantities, profit and cost.
2. As the `unit cost` of the product remains `same` in all the duplicate records, most likely reason could be:
     - The customer might have updated the cart with some quantity of a product and the again added the same
        product with a different quantity.

In [5822]:
too_many_dups = {}
dups = df[df['Order ID'].duplicated()].sort_values('Order ID')

In [5824]:
for i in dups['Order ID']:
    too_many_dups[i] = too_many_dups.get(i, 0) + 1

In [5826]:
many_dups = []
for i in too_many_dups.items():
    if i[1] > 1:
        many_dups.append(i[0])
        

In [5828]:
df[df['Order ID'].duplicated()].sort_values(['Order ID', 'Product Name'])[['Order ID', 'Product Name', 'Sales', 
                                                                           'Units', 'Gross Profit', 'Cost']] \
                                    .style.apply(
                                           highlight_rows, colour=random.choice(colours), 
                                           column_name='Order ID',
                                           column_values = many_dups,
                                           axis=1)

,Order ID,Product Name,Sales,Units,Gross Profit,Cost
64,CA-2021-115238-CHO-FUD-51000,Wonka Bar - Fudge Mallows,3.600000,1,2.400000,1.200000
65,CA-2021-115238-CHO-MIL-31000,Wonka Bar - Milk Chocolate,32.500000,10,21.100000,11.400000
2019,CA-2021-123625-CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.500000,2,4.220000,2.280000
2022,CA-2021-123625-CHO-MIL-31000,Wonka Bar - Milk Chocolate,16.250000,5,10.550000,5.700000
1298,CA-2021-131807-CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,3.600000,1,2.500000,1.100000
2021,CA-2021-141531-CHO-MIL-31000,Wonka Bar - Milk Chocolate,16.250000,5,10.550000,5.700000
1871,CA-2021-151799-CHO-MIL-31000,Wonka Bar - Milk Chocolate,3.250000,1,2.110000,1.140000
1872,CA-2021-151799-CHO-MIL-31000,Wonka Bar - Milk Chocolate,9.750000,3,6.330000,3.420000
516,CA-2021-152443-CHO-MIL-31000,Wonka Bar - Milk Chocolate,9.750000,3,6.330000,3.420000
1214,CA-2021-160325-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.470000,3,7.470000,3.000000


### Each Order ID only consists of one unique product
- This confirms the above hypothesis. 

In [5831]:
x = df.groupby(['Order ID']).agg(count=('Product Name', 'nunique')).reset_index()

In [5833]:
x

,Order ID,count
0,CA-2021-100867-CHO-FUD-51000,1
1,CA-2021-107153-CHO-FUD-51000,1
2,CA-2021-115238-CHO-FUD-51000,1
3,CA-2021-115238-CHO-MIL-31000,1
4,CA-2021-115238-CHO-TRI-54000,1
...,...,...
8544,US-2024-169901-CHO-NUT-13000,1
8545,US-2024-169929-CHO-TRI-54000,1
8546,US-2024-169978-CHO-TRI-54000,1
8547,US-2024-169999-CHO-SCR-58000,1


In [5835]:
(x['count'] > 1).sum()

0

In [5837]:
df.columns

Index(['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID',
       'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division',
       'Region', 'Product ID', 'Product Name', 'Sales', 'Units',
       'Gross Profit', 'Cost'],
      dtype='object')

### This would imply that the only changing factors across duplicated rows are:
1. sales
2. cost
3. gross profit
4. units

In [5840]:
df.groupby(['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID',
       'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division',
       'Region', 'Product ID', 'Product Name']).size()

Order ID                      Order Date  Ship Date   Ship Mode       Customer ID  Country/Region  City           State/Province    Postal Code  Division   Region    Product ID     Product Name                     
CA-2021-100867-CHO-FUD-51000  19-10-2024  16-04-2027  Standard Class  100867       Canada          Vancouver      British Columbia  V6G          Chocolate  Pacific   CHO-FUD-51000  Wonka Bar - Fudge Mallows            1
CA-2021-107153-CHO-FUD-51000  28-09-2024  26-03-2027  Standard Class  107153       Canada          Montreal       Quebec            H1A          Chocolate  Atlantic  CHO-FUD-51000  Wonka Bar - Fudge Mallows            1
CA-2021-115238-CHO-FUD-51000  21-01-2024  17-07-2026  Standard Class  115238       Canada          Toronto        Ontario           M7A          Chocolate  Atlantic  CHO-FUD-51000  Wonka Bar - Fudge Mallows            2
CA-2021-115238-CHO-MIL-31000  21-01-2024  17-07-2026  Standard Class  115238       Canada          Toronto        Ontario    

### The best approach to handle the duplicated Order IDs would be to group them and aggregate the quantities, such as :`sales`, `units`, `gross profit` and `cost`.
    

In [5843]:
dframe = df.groupby('Order ID').agg(
    order_date = ('Order Date', 'max'),
    ship_date = ('Ship Date', 'max'),
    ship_mode = ('Ship Mode', 'max'),
    customer_id = ('Customer ID', 'max'),
    country_customer = ('Country/Region', 'max'),
    city_customer = ('City', 'max'),
    state_customer = ('State/Province', 'max'),
    postal_code_customer = ('Postal Code', 'max'),
    division = ('Division', 'max'),
    region_customer = ('Region', 'max'),
    product_id = ('Product ID', 'max'),
    product_name = ('Product Name', 'max'),
    sales = ('Sales', 'sum'),
    units = ('Units', 'sum'),
    gross_profit = ('Gross Profit', 'sum'),
    cost= ('Cost', 'sum')
    
    
).reset_index()

In [5844]:
dframe

,Order ID,order_date,ship_date,ship_mode,customer_id,country_customer,city_customer,state_customer,postal_code_customer,division,region_customer,product_id,product_name,sales,units,gross_profit,cost
0,CA-2021-100867-CHO-FUD-51000,19-10-2024,16-04-2027,Standard Class,100867,Canada,Vancouver,British Columbia,V6G,Chocolate,Pacific,CHO-FUD-51000,Wonka Bar - Fudge Mallows,10.80,3,7.20,3.60
1,CA-2021-107153-CHO-FUD-51000,28-09-2024,26-03-2027,Standard Class,107153,Canada,Montreal,Quebec,H1A,Chocolate,Atlantic,CHO-FUD-51000,Wonka Bar - Fudge Mallows,7.20,2,4.80,2.40
2,CA-2021-115238-CHO-FUD-51000,21-01-2024,17-07-2026,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-FUD-51000,Wonka Bar - Fudge Mallows,21.60,6,14.40,7.20
3,CA-2021-115238-CHO-MIL-31000,21-01-2024,17-07-2026,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,35.75,11,23.21,12.54
4,CA-2021-115238-CHO-TRI-54000,21-01-2024,17-07-2026,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,18.75,5,12.25,6.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8544,US-2024-169901-CHO-NUT-13000,15-06-2025,10-12-2029,Standard Class,169901,United States,San Francisco,California,94122,Chocolate,Pacific,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
8545,US-2024-169929-CHO-TRI-54000,25-09-2025,21-03-2030,First Class,169929,United States,Helena,Montana,59601,Chocolate,Pacific,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,18.75,5,12.25,6.50
8546,US-2024-169978-CHO-TRI-54000,24-12-2025,20-06-2030,Standard Class,169978,United States,Oceanside,New York,11572,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,18.75,5,12.25,6.50
8547,US-2024-169999-CHO-SCR-58000,19-06-2025,17-12-2029,Standard Class,169999,United States,New York City,New York,10035,Chocolate,Atlantic,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30


### Converting the string dates to datetime object

In [5846]:
dframe['ship_date'] = dframe['ship_date'].apply(lambda x: datetime.strptime(x, "%d-%m-%Y"))

In [5847]:
dframe['order_date'] = dframe['order_date'].apply(lambda x: datetime.strptime(x, "%d-%m-%Y"))

In [5848]:
dframe['lead_time'] =  dframe['ship_date'] - dframe['order_date']

In [5849]:
dframe['lead_time'] =  dframe['lead_time'].apply(lambda x: x.days)

In [5850]:
dframe.head()

,Order ID,order_date,ship_date,ship_mode,customer_id,country_customer,city_customer,state_customer,postal_code_customer,division,region_customer,product_id,product_name,sales,units,gross_profit,cost,lead_time
0,CA-2021-100867-CHO-FUD-51000,2024-10-19,2027-04-16,Standard Class,100867,Canada,Vancouver,British Columbia,V6G,Chocolate,Pacific,CHO-FUD-51000,Wonka Bar - Fudge Mallows,10.80,3,7.20,3.60,909
1,CA-2021-107153-CHO-FUD-51000,2024-09-28,2027-03-26,Standard Class,107153,Canada,Montreal,Quebec,H1A,Chocolate,Atlantic,CHO-FUD-51000,Wonka Bar - Fudge Mallows,7.20,2,4.80,2.40,909
2,CA-2021-115238-CHO-FUD-51000,2024-01-21,2026-07-17,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-FUD-51000,Wonka Bar - Fudge Mallows,21.60,6,14.40,7.20,908
3,CA-2021-115238-CHO-MIL-31000,2024-01-21,2026-07-17,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-MIL-31000,Wonka Bar - Milk Chocolate,35.75,11,23.21,12.54,908
4,CA-2021-115238-CHO-TRI-54000,2024-01-21,2026-07-17,Standard Class,115238,Canada,Toronto,Ontario,M7A,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,18.75,5,12.25,6.50,908


In [5851]:
dframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8549 entries, 0 to 8548
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Order ID              8549 non-null   object        
 1   order_date            8549 non-null   datetime64[ns]
 2   ship_date             8549 non-null   datetime64[ns]
 3   ship_mode             8549 non-null   object        
 4   customer_id           8549 non-null   int64         
 5   country_customer      8549 non-null   object        
 6   city_customer         8549 non-null   object        
 7   state_customer        8549 non-null   object        
 8   postal_code_customer  8549 non-null   object        
 9   division              8549 non-null   object        
 10  region_customer       8549 non-null   object        
 11  product_id            8549 non-null   object        
 12  product_name          8549 non-null   object        
 13  sales             

### Found no irregularities in the `Order date` column.

In [5861]:
dframe['order_date'].max()

Timestamp('2025-12-31 00:00:00')

In [5863]:
dframe['order_date'].min()

Timestamp('2024-01-02 00:00:00')

In [5865]:
dframe['order_date'].isnull().sum()

0

### The `ship dates` columns seems to be containing unreliable values, as the dates are going up by `several years`.

In [5995]:
pd.DataFrame([{'min_ship_date': dframe['ship_date'].min()}]).style.apply(lambda x: [f'background-color : {random.choice(colours)}']*len(x))

,min_ship_date
0,2026-06-30 00:00:00


In [5999]:
pd.DataFrame([{'max_ship_date': dframe['ship_date'].max()}]).style.apply(lambda x: [f'background-color : {random.choice(colours)}']*len(x))

,max_ship_date
0,2030-06-28 00:00:00


### Ship mode

In [5873]:
dframe['ship_mode'].unique()

array(['Standard Class', 'First Class', 'Second Class', 'Same Day'],
      dtype=object)

In [5875]:
dframe['ship_mode'].value_counts()

ship_mode
Standard Class    5142
Second Class      1653
First Class       1312
Same Day           442
Name: count, dtype: int64

### Standardizing the Product Name column.

In [5878]:
dframe['product_name'].unique()

array(['Wonka Bar - Fudge Mallows', 'Wonka Bar - Milk Chocolate',
       'Wonka Bar - Triple Dazzle Caramel',
       'Wonka Bar - Nutty Crunch Surprise',
       'Wonka Bar -Scrumdiddlyumptious', 'Lickable Wallpaper',
       'Wonka Gum', 'Kazookles', 'SweeTARTS', 'Fizzy Lifting Drinks',
       'Laffy Taffy', 'Nerds', 'Everlasting Gobstopper', 'Fun Dip',
       'Hair Toffee'], dtype=object)

In [5880]:
dframe['product_name'].nunique()

15

In [5882]:
dframe['product_name'].apply(lambda x: preprocessing_names(x)).nunique()

15

In [5884]:
dframe['product_name'] =  dframe['product_name'].apply(lambda x: preprocessing_names(x))

In [5886]:
dframe['product_name'].unique()

array(['wonka bar fudge mallows', 'wonka bar milk chocolate',
       'wonka bar triple dazzle caramel',
       'wonka bar nutty crunch surprise', 'wonka bar scrumdiddlyumptious',
       'lickable wallpaper', 'wonka gum', 'kazookles', 'sweetarts',
       'fizzy lifting drinks', 'laffy taffy', 'nerds',
       'everlasting gobstopper', 'fun dip', 'hair toffee'], dtype=object)

### Standardizing the country_customer column.

In [5889]:
dframe['country_customer'].unique()

array(['Canada', 'United States'], dtype=object)

In [5891]:
dframe['country_customer'] = dframe['country_customer'].apply(lambda x: preprocessing_names(x))

### Standardizing the city_customer column.

In [5894]:
dframe['city_customer'].nunique()

542

In [5896]:
dframe['city_customer'].apply(lambda x: preprocessing_names(x)).nunique()

542

In [5898]:
dframe['city_customer'] =  dframe['city_customer'].apply(lambda x: preprocessing_names(x))

### Standardizing the state_customer column.

In [5901]:
dframe['state_customer'].nunique()

59

In [5902]:
dframe['state_customer'].apply(lambda x: preprocessing_names(x)).nunique()

59

In [5905]:
dframe['state_customer'] = dframe['state_customer'].apply(lambda x: preprocessing_names(x))

### Standardizing the region_customer column.

In [5908]:
dframe['region_customer'].nunique()

4

In [5910]:
dframe['region_customer'] = dframe['region_customer'].apply(lambda x: preprocessing_names(x))

### `Lead Time` Correction ‼️ ------  IMPORTANT  ------- ‼️

In [5913]:
dframe.lead_time

0        909
1        909
2        908
3        908
4        908
        ... 
8544    1639
8545    1638
8546    1639
8547    1642
8548    1642
Name: lead_time, Length: 8549, dtype: int64

In [5915]:
dframe.lead_time.min(), dframe.lead_time.max()

(904, 1642)

### As the `lead time` could only reach 15 to 16 days due to the `transfer items`, it would only make sense to cap the values at 16

In [5918]:
dframe['lead_time'] =  round(dframe.lead_time/100).astype(int)

In [5920]:
(dframe.loc[0]['ship_date'] - dframe.loc[0]['order_date']).days

909

In [5922]:
dframe.head(3)[['Order ID', 'order_date', 'ship_date', 'lead_time']]

,Order ID,order_date,ship_date,lead_time
0,CA-2021-100867-CHO-FUD-51000,2024-10-19,2027-04-16,9
1,CA-2021-107153-CHO-FUD-51000,2024-09-28,2027-03-26,9
2,CA-2021-115238-CHO-FUD-51000,2024-01-21,2026-07-17,9


### Standardizing the factories table.

In [5925]:
df_factories

,Factory,Latitude,Longitude,City,State,Country
0,Lot's O' Nuts,32.881893,-111.768036,Casa Grande,AZ,USA
1,Wicked Choccy's,32.076176,-81.088371,Savannah,GA,USA
2,Sugar Shack,48.119140,-96.181150,Thief River Falls,MN,USA
3,Secret Factory,41.446333,-90.565487,Moline,IL,USA
4,The Other Factory,35.117500,-89.971107,Memphis,TN,USA


In [5927]:
df_factories['Factory'] = df_factories['Factory'].apply(lambda x: preprocessing_names(x))
df_factories['City'] = df_factories['City'].apply(lambda x: preprocessing_names(x))
df_factories['State'] = df_factories['State'].apply(lambda x: preprocessing_names(x))
df_factories['Country'] = df_factories['Country'].apply(lambda x: preprocessing_names(x))

In [5929]:
df_factories

,Factory,Latitude,Longitude,City,State,Country
0,lots o nuts,32.881893,-111.768036,casa grande,az,usa
1,wicked choccys,32.076176,-81.088371,savannah,ga,usa
2,sugar shack,48.119140,-96.181150,thief river falls,mn,usa
3,secret factory,41.446333,-90.565487,moline,il,usa
4,the other factory,35.117500,-89.971107,memphis,tn,usa


### Standardizing the products table.

In [5932]:
df_products

,Division,Product Name,Factory
0,Chocolate,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
1,Chocolate,Wonka Bar - Fudge Mallows,Lot's O' Nuts
2,Chocolate,Wonka Bar - Scrumdiddlyumptious,Lot's O' Nuts
3,Chocolate,Wonka Bar - Milk Chocolate,Wicked Choccy's
4,Chocolate,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
5,Sugar,Laffy Taffy,Sugar Shack
6,Sugar,SweeTARTS,Sugar Shack
7,Sugar,Nerds,Sugar Shack
8,Sugar,Fun Dip,Sugar Shack
9,Other,Fizzy Lifting Drinks,Sugar Shack


In [5934]:
df_products['Product Name'] = df_products['Product Name'].apply(lambda x: preprocessing_names(x))
df_products['Factory'] = df_products['Factory'].apply(lambda x: preprocessing_names(x))

In [5936]:
df_products.rename(columns={'Division':'division', 'Product Name': 'product_name', 'Factory': 'factory' }, inplace=True)

### In the `products table`:
1. `fizzy lifting drink` is incorrectly classified as `Other` division.
2. But since the product were produced by 'sugar shack' factory, it would make sense to put it in the same division.
3. the said product is in the `sugar` division in the actual data, which further validates the reasoning. 

In [5939]:
df_products.style.apply(highlight_rows, colour=random.choice(colours), 
                                       column_name='product_name',
                                       column_values = ['fizzy lifting drinks'],
                                       axis=1)

,division,product_name,factory
0,Chocolate,wonka bar nutty crunch surprise,lots o nuts
1,Chocolate,wonka bar fudge mallows,lots o nuts
2,Chocolate,wonka bar scrumdiddlyumptious,lots o nuts
3,Chocolate,wonka bar milk chocolate,wicked choccys
4,Chocolate,wonka bar triple dazzle caramel,wicked choccys
5,Sugar,laffy taffy,sugar shack
6,Sugar,sweetarts,sugar shack
7,Sugar,nerds,sugar shack
8,Sugar,fun dip,sugar shack
9,Other,fizzy lifting drinks,sugar shack


### Original data

In [5942]:
dframe.groupby(['division', 'product_name'
               ]).size().reset_index().style.apply(highlight_rows, 
                                                   colour=random.choice(colours), 
                                                   column_name='product_name',
                                                   column_values = ['fizzy lifting drinks'],
                                                   axis=1)

,division,product_name,0
0,Chocolate,wonka bar fudge mallows,1527
1,Chocolate,wonka bar milk chocolate,1768
2,Chocolate,wonka bar nutty crunch surprise,1529
3,Chocolate,wonka bar scrumdiddlyumptious,1704
4,Chocolate,wonka bar triple dazzle caramel,1677
5,Other,kazookles,94
6,Other,lickable wallpaper,92
7,Other,wonka gum,118
8,Sugar,everlasting gobstopper,3
9,Sugar,fizzy lifting drinks,6


### Renaming the `division` value for `fizzy lifting drinks` from `other` to `Sugar` in products table.

In [5945]:
df_products.loc[df_products['product_name'] == 'fizzy lifting drinks', 'division'] = 'Sugar'

In [5947]:
df_products

,division,product_name,factory
0,Chocolate,wonka bar nutty crunch surprise,lots o nuts
1,Chocolate,wonka bar fudge mallows,lots o nuts
2,Chocolate,wonka bar scrumdiddlyumptious,lots o nuts
3,Chocolate,wonka bar milk chocolate,wicked choccys
4,Chocolate,wonka bar triple dazzle caramel,wicked choccys
5,Sugar,laffy taffy,sugar shack
6,Sugar,sweetarts,sugar shack
7,Sugar,nerds,sugar shack
8,Sugar,fun dip,sugar shack
9,Sugar,fizzy lifting drinks,sugar shack


In [5949]:
dframe =  dframe.merge(right=df_products, how='inner', on=['product_name', 'division'])

In [5951]:
dframe.shape

(8549, 19)

In [5953]:
dframe.isnull().sum()

Order ID                0
order_date              0
ship_date               0
ship_mode               0
customer_id             0
country_customer        0
city_customer           0
state_customer          0
postal_code_customer    0
division                0
region_customer         0
product_id              0
product_name            0
sales                   0
units                   0
gross_profit            0
cost                    0
lead_time               0
factory                 0
dtype: int64

### Factories table

In [5956]:
df_factories

,Factory,Latitude,Longitude,City,State,Country
0,lots o nuts,32.881893,-111.768036,casa grande,az,usa
1,wicked choccys,32.076176,-81.088371,savannah,ga,usa
2,sugar shack,48.119140,-96.181150,thief river falls,mn,usa
3,secret factory,41.446333,-90.565487,moline,il,usa
4,the other factory,35.117500,-89.971107,memphis,tn,usa


In [5958]:
df_factories.rename(columns={'Factory':'factory', 'Latitude':'lat_factory', 'Longitude':'long_company', 
                             'City':'city_company', 'State':'state_company', 'Country':'country_company'}, inplace=True)

In [5960]:
df_factories

,factory,lat_factory,long_company,city_company,state_company,country_company
0,lots o nuts,32.881893,-111.768036,casa grande,az,usa
1,wicked choccys,32.076176,-81.088371,savannah,ga,usa
2,sugar shack,48.119140,-96.181150,thief river falls,mn,usa
3,secret factory,41.446333,-90.565487,moline,il,usa
4,the other factory,35.117500,-89.971107,memphis,tn,usa


In [5961]:
dframe =  dframe.merge(df_factories, how='inner', on='factory')

In [5964]:
dframe.shape

(8549, 24)

In [5966]:
preprocessed_df =  dframe.copy()

In [7147]:
preprocessed_df.to_csv("preprocessed_df.csv", index=False)

In [7282]:
preprocessed_df

,Order ID,order_date,ship_date,ship_mode,customer_id,country_customer,city_customer,state_customer,postal_code_customer,division,region_customer,product_id,product_name,sales,units,gross_profit,cost,lead_time,factory,lat_factory,long_company,city_company,state_company,country_company
0,CA-2021-100867-CHO-FUD-51000,2024-10-19,2027-04-16,Standard Class,100867,canada,vancouver,british columbia,V6G,Chocolate,pacific,CHO-FUD-51000,wonka bar fudge mallows,10.80,3,7.20,3.60,9,lots o nuts,32.881893,-111.768036,casa grande,az,usa
1,CA-2021-107153-CHO-FUD-51000,2024-09-28,2027-03-26,Standard Class,107153,canada,montreal,quebec,H1A,Chocolate,atlantic,CHO-FUD-51000,wonka bar fudge mallows,7.20,2,4.80,2.40,9,lots o nuts,32.881893,-111.768036,casa grande,az,usa
2,CA-2021-115238-CHO-FUD-51000,2024-01-21,2026-07-17,Standard Class,115238,canada,toronto,ontario,M7A,Chocolate,atlantic,CHO-FUD-51000,wonka bar fudge mallows,21.60,6,14.40,7.20,9,lots o nuts,32.881893,-111.768036,casa grande,az,usa
3,CA-2021-115238-CHO-MIL-31000,2024-01-21,2026-07-17,Standard Class,115238,canada,toronto,ontario,M7A,Chocolate,atlantic,CHO-MIL-31000,wonka bar milk chocolate,35.75,11,23.21,12.54,9,wicked choccys,32.076176,-81.088371,savannah,ga,usa
4,CA-2021-115238-CHO-TRI-54000,2024-01-21,2026-07-17,Standard Class,115238,canada,toronto,ontario,M7A,Chocolate,atlantic,CHO-TRI-54000,wonka bar triple dazzle caramel,18.75,5,12.25,6.50,9,wicked choccys,32.076176,-81.088371,savannah,ga,usa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8544,US-2024-169901-CHO-NUT-13000,2025-06-15,2029-12-10,Standard Class,169901,united states,san francisco,california,94122,Chocolate,pacific,CHO-NUT-13000,wonka bar nutty crunch surprise,10.47,3,7.47,3.00,16,lots o nuts,32.881893,-111.768036,casa grande,az,usa
8545,US-2024-169929-CHO-TRI-54000,2025-09-25,2030-03-21,First Class,169929,united states,helena,montana,59601,Chocolate,pacific,CHO-TRI-54000,wonka bar triple dazzle caramel,18.75,5,12.25,6.50,16,wicked choccys,32.076176,-81.088371,savannah,ga,usa
8546,US-2024-169978-CHO-TRI-54000,2025-12-24,2030-06-20,Standard Class,169978,united states,oceanside,new york,11572,Chocolate,atlantic,CHO-TRI-54000,wonka bar triple dazzle caramel,18.75,5,12.25,6.50,16,wicked choccys,32.076176,-81.088371,savannah,ga,usa
8547,US-2024-169999-CHO-SCR-58000,2025-06-19,2029-12-17,Standard Class,169999,united states,new york city,new york,10035,Chocolate,atlantic,CHO-SCR-58000,wonka bar scrumdiddlyumptious,10.80,3,7.50,3.30,16,lots o nuts,32.881893,-111.768036,casa grande,az,usa


## Analysis: 

### Overall Performance of different `Factories`.

In [6020]:
p = preprocessed_df.groupby('factory').agg(
                                    total_shipments=('Order ID', 'count'),
                                    avg_lead_time=('lead_time', 'mean'),
                                    avg_lead_time_variability=('lead_time', 'std'),
                                    total_gross_profit=('gross_profit', 'sum'),
                                    total_cost=('cost', 'sum'),
                                    
                                            )
p.sort_values('total_gross_profit', ascending=False).reset_index()

,factory,total_shipments,avg_lead_time,avg_lead_time_variability,total_gross_profit,total_cost
0,lots o nuts,4760,13.193067,2.472610,51347.69,22940.20
1,wicked choccys,3445,13.206967,2.484296,35110.54,18794.96
2,secret factory,213,13.140845,2.745300,4344.70,4242.80
3,the other factory,98,12.857143,2.487080,152.25,1130.00
4,sugar shack,33,13.363636,2.497726,121.23,99.75


### Overall performance of different `shipping methods`.

In [7290]:
preprocessed_df.groupby(['ship_mode']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('avg_lead_time', ascending=False).reset_index()

,ship_mode,total_shipments,avg_lead_time,avg_lead_variability
0,First Class,1312,13.349,2.503
1,Same Day,442,13.303,2.416
2,Second Class,1653,13.196,2.468
3,Standard Class,5142,13.145,2.489


### Shipping performance across factories

In [7552]:
ship_factory = preprocessed_df.groupby(['ship_mode', 'factory']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('avg_lead_time', ascending=False).reset_index()

In [7554]:
y = ship_factory['total_shipments'].sort_values().tolist()
q(y, [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9])

{'10%': [5, 18.0, 2.0],
 '20%': [11, 16.0, 4.0],
 '25%': [17, 15.0, 5.0],
 '30%': [21, 14.0, 6.0],
 '35%': [23, 13.0, 7.0],
 '40%': [42, 12.0, 8.0],
 '45%': [47, 11.0, 9.0],
 '50%': [53, 10.0, 10.0],
 '60%': [183, 8.0, 12.0],
 '70%': [499, 6.0, 14.0],
 '80%': [745, 4.0, 16.0],
 '90%': [2110, 2.0, 18.0]}

In [7558]:
ship_factory =  ship_factory[ship_factory['total_shipments'] > 11 ]
ship_factory.sort_values('factory')

,ship_mode,factory,total_shipments,avg_lead_time,avg_lead_variability
3,Same Day,lots o nuts,240,13.404,2.346
6,First Class,lots o nuts,745,13.358,2.478
12,Second Class,lots o nuts,932,13.153,2.462
13,Standard Class,lots o nuts,2843,13.145,2.483
2,Second Class,secret factory,42,13.690,2.533
14,First Class,secret factory,47,13.021,2.930
15,Standard Class,secret factory,113,12.965,2.764
9,Standard Class,sugar shack,23,13.174,2.622
16,First Class,the other factory,17,12.941,2.609
17,Standard Class,the other factory,53,12.830,2.432


### Performance of `shipping methods` across different `customer-regions`.

In [6795]:
ship_region_grouped = preprocessed_df.groupby(['ship_mode', 'region_customer']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('region_customer', ascending=True).reset_index()
ship_region_grouped

,ship_mode,region_customer,total_shipments,avg_lead_time,avg_lead_variability
0,First Class,atlantic,407,13.287,2.440
1,Same Day,atlantic,126,13.040,2.391
2,Second Class,atlantic,456,13.259,2.412
3,Standard Class,atlantic,1487,13.202,2.448
4,First Class,gulf,201,13.214,2.600
5,Same Day,gulf,65,12.862,2.474
6,Second Class,gulf,276,13.170,2.587
7,Standard Class,gulf,808,13.069,2.484
8,First Class,interior,273,13.407,2.632
9,Same Day,interior,98,13.837,2.237


In [7528]:
y = ship_region_grouped['total_shipments'].sort_values().tolist()
q(y, [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9])

{'10%': [126, 14.0, 2.0],
 '20%': [201, 12.0, 4.0],
 '25%': [201, 12.0, 4.0],
 '30%': [273, 11.0, 5.0],
 '35%': [276, 10.0, 6.0],
 '40%': [378, 9.0, 7.0],
 '45%': [407, 8.0, 8.0],
 '50%': [407, 8.0, 8.0],
 '60%': [456, 6.0, 10.0],
 '70%': [808, 4.0, 12.0],
 '80%': [1230, 3.0, 13.0],
 '90%': [1617, 1.0, 15.0]}

### The average value for `lead time` could become unreliable if the `total shipments` are too low.
- Better approach to handle such cases would be to use `confidence intervals`.
- The `confidence column` would tell us how reliable the `average lead times` are.

In [6798]:
ship_region_grouped['confidence'] =  ship_region_grouped.apply(lambda x: \
                                                            'good' if x['total_shipments']>=30 else \
                                                            'medium' if x['total_shipments'] >= 20 else \
                                                            'weak'
                                                            ,axis=1
                                                        )                    

In [6800]:
ship_region_grouped['confidence'].value_counts()

confidence
good    16
Name: count, dtype: int64

### Using `upper bound` in CI for `average lead time`, as we want to take the worst case scenario into consideration.

In [6803]:
ship_region_grouped['avg_lead_time'] = \
                                ship_region_grouped.apply(lambda x: \
                                                            round(z_upper_bound(x['avg_lead_time'], 
                                                            x['avg_lead_variability'], 
                                                            x['total_shipments'] ), 3) if x['confidence'] == 'good' else \
                                                               
                                                           round(t_upper_bound(x['avg_lead_time'],
                                                                        x['avg_lead_variability'],
                                                                        x['total_shipments']), 3) 
                                                         , axis = 1)
ship_region_grouped.reset_index(drop=True)

,ship_mode,region_customer,total_shipments,avg_lead_time,avg_lead_variability,confidence
0,First Class,atlantic,407,13.524,2.440,good
1,Same Day,atlantic,126,13.457,2.391,good
2,Second Class,atlantic,456,13.480,2.412,good
3,Standard Class,atlantic,1487,13.326,2.448,good
4,First Class,gulf,201,13.573,2.600,good
5,Same Day,gulf,65,13.463,2.474,good
6,Second Class,gulf,276,13.475,2.587,good
7,Standard Class,gulf,808,13.240,2.484,good
8,First Class,interior,273,13.719,2.632,good
9,Same Day,interior,98,14.280,2.237,good


## Engineering feature `efficiency_score` which depends on:
1. normalized average lead time
2. normalized average lead time variability
3. For now, I'm providing them equal weightage.

In [6806]:
## normalized average lead time
ship_region_grouped.insert(5, column='norm_lead_time', value = 
            round((ship_region_grouped['avg_lead_time'] - ship_region_grouped['avg_lead_time'].min()) / \
            (ship_region_grouped['avg_lead_time'].max() - ship_region_grouped['avg_lead_time'].min()), 3))

## normalized average lead time variability
ship_region_grouped.insert(6, column='norm_variability', value = 
            round((ship_region_grouped['avg_lead_variability'] - ship_region_grouped['avg_lead_variability'].min()) / \
            (ship_region_grouped['avg_lead_variability'].max() - ship_region_grouped['avg_lead_variability'].min()),3 ))

## efficiency score
ship_region_grouped.insert(7, column='efficiency_score', value = 
                round(1 - ((0.5 * ship_region_grouped['norm_lead_time']) + (0.5 * ship_region_grouped['norm_variability'])), 3))

In [7280]:
ship_region_grouped

,ship_mode,region_customer,total_shipments,avg_lead_time,avg_lead_variability,norm_lead_time,norm_variability,efficiency_score,confidence
0,First Class,atlantic,407,13.524,2.440,0.275,0.514,0.605,good
1,Same Day,atlantic,126,13.457,2.391,0.211,0.390,0.700,good
2,Second Class,atlantic,456,13.480,2.412,0.233,0.443,0.662,good
3,Standard Class,atlantic,1487,13.326,2.448,0.085,0.534,0.690,good
4,First Class,gulf,201,13.573,2.600,0.322,0.919,0.379,good
5,Same Day,gulf,65,13.463,2.474,0.217,0.600,0.592,good
6,Second Class,gulf,276,13.475,2.587,0.228,0.886,0.443,good
7,Standard Class,gulf,808,13.240,2.484,0.003,0.625,0.686,good
8,First Class,interior,273,13.719,2.632,0.462,1.000,0.269,good
9,Same Day,interior,98,14.280,2.237,1.000,0.000,0.500,good


### The best `shipping methods` across all the `customer-regions` with the `lowest average lead time`

In [6809]:
ship_region_grouped.loc[ship_region_grouped.groupby('region_customer')['efficiency_score'].idxmax()].reset_index(drop=True)

,ship_mode,region_customer,total_shipments,avg_lead_time,avg_lead_variability,norm_lead_time,norm_variability,efficiency_score,confidence
0,Same Day,atlantic,126,13.457,2.391,0.211,0.390,0.700,good
1,Standard Class,gulf,808,13.240,2.484,0.003,0.625,0.686,good
2,Second Class,interior,378,13.342,2.390,0.101,0.387,0.756,good
3,Standard Class,pacific,1617,13.237,2.525,0.000,0.729,0.636,good


### Performance across different `shipping methods` across different `states`.

In [7492]:
ship_state_grouped = preprocessed_df.groupby(['ship_mode', 'state_customer']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('state_customer', ascending=True).reset_index().fillna(0)
ship_state_grouped

,ship_mode,state_customer,total_shipments,avg_lead_time,avg_lead_variability
0,First Class,alabama,7,12.857,2.035
1,Second Class,alabama,15,13.133,2.900
2,Standard Class,alabama,30,12.900,2.339
3,Same Day,alabama,1,13.000,0.000
4,Second Class,alberta,12,12.833,2.125
...,...,...,...,...,...
190,First Class,wisconsin,12,12.333,3.172
191,Second Class,wisconsin,11,13.000,0.000
192,Same Day,wisconsin,5,14.800,1.643
193,Standard Class,wisconsin,63,13.508,2.109


### `Customer-states` that have only `1 shipment` for a specific shipping method. 
- Better to filter them out as noise as we cannot rely on their average lead time.

In [7495]:
ship_state_grouped[ship_state_grouped['total_shipments']==1]

,ship_mode,state_customer,total_shipments,avg_lead_time,avg_lead_variability
3,Same Day,alabama,1,13.0,0.0
31,Same Day,delaware,1,13.0,0.0
54,Same Day,iowa,1,13.0,0.0
55,First Class,iowa,1,13.0,0.0
59,Same Day,kansas,1,16.0,0.0
96,First Class,montana,1,16.0,0.0
98,Same Day,montana,1,9.0,0.0
103,Same Day,nevada,1,9.0,0.0
114,Same Day,new jersey,1,16.0,0.0
115,Same Day,new mexico,1,9.0,0.0


### Setting the `threshold value` = 12, which equals ~40 percentile in `total shipments` across state-wise shipping-methods.

In [7498]:
y = ship_state_grouped['total_shipments'].sort_values().tolist()
q(y, [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9])

{'20%': [4, 156.0, 39.0],
 '25%': [4, 146.0, 49.0],
 '30%': [6, 136.0, 59.0],
 '35%': [8, 126.0, 69.0],
 '40%': [10, 117.0, 78.0],
 '45%': [12, 107.0, 88.0],
 '50%': [14, 97.0, 98.0],
 '60%': [18, 78.0, 117.0],
 '70%': [30, 58.0, 137.0],
 '80%': [52, 39.0, 156.0],
 '90%': [98, 19.0, 176.0]}

In [7500]:
threshold = 12

In [7502]:
ship_state_grouped =  ship_state_grouped[ship_state_grouped['total_shipments'] >= 12]
ship_state_grouped

,ship_mode,state_customer,total_shipments,avg_lead_time,avg_lead_variability
1,Second Class,alabama,15,13.133,2.900
2,Standard Class,alabama,30,12.900,2.339
4,Second Class,alberta,12,12.833,2.125
6,First Class,arizona,31,12.581,2.802
7,Second Class,arizona,24,13.542,1.956
...,...,...,...,...,...
185,Second Class,washington,94,13.426,2.495
186,Same Day,washington,32,13.781,2.511
187,First Class,washington,56,14.179,2.601
190,First Class,wisconsin,12,12.333,3.172


### Creating the `confidence column`.

In [7505]:
ship_state_grouped.insert(5, column='confidence', value=
                           ship_state_grouped.apply(lambda x: 'good' if x['total_shipments']>=30 else \
                                                               'medium' if x['total_shipments'] >= 10 else \
                                                               'weak'
                                                               , axis=1)
                         ) 

In [7507]:
ship_state_grouped['confidence'].value_counts()

confidence
good      59
medium    48
Name: count, dtype: int64

### Using `upper_bound_CI` for `average lead time`.

In [7514]:
ship_state_grouped.loc[:, 'avg_lead_time'] = \
                                ship_state_grouped.apply(lambda x: \
                                                            round(z_upper_bound(x['avg_lead_time'], 
                                                            x['avg_lead_variability'], 
                                                            x['total_shipments'] ), 3) if x['confidence'] == 'good' else \
                                                               
                                                           round(t_upper_bound(x['avg_lead_time'],
                                                                        x['avg_lead_variability'],
                                                                        x['total_shipments']), 3)
                                                         , axis = 1)

ship_state_grouped.reset_index(drop=True, inplace=True)

### Efficiency score

In [7517]:
## normalized average lead time
ship_state_grouped.insert(6, column='norm_lead_time', value = 
            round((ship_state_grouped['avg_lead_time'] - ship_state_grouped['avg_lead_time'].min()) / \
            (ship_state_grouped['avg_lead_time'].max() - ship_state_grouped['avg_lead_time'].min()), 3))

## normalized average lead time variability
ship_state_grouped.insert(7, column='norm_variability', value = 
            round((ship_state_grouped['avg_lead_variability'] - ship_state_grouped['avg_lead_variability'].min()) / \
            (ship_state_grouped['avg_lead_variability'].max() - ship_state_grouped['avg_lead_variability'].min()),3 ))

## efficiency score
ship_state_grouped.insert(8, column='efficiency_score', value = 
                round(1 - ((0.5 * ship_state_grouped['norm_lead_time']) + (0.5 * ship_state_grouped['norm_variability'])), 3))

In [7519]:
ship_state_grouped

,ship_mode,state_customer,total_shipments,avg_lead_time,avg_lead_variability,confidence,norm_lead_time,norm_variability,efficiency_score
0,Second Class,alabama,15,17.951,2.900,medium,0.908,0.833,0.129
1,Standard Class,alabama,30,15.411,2.339,good,0.386,0.607,0.504
2,Second Class,alberta,12,16.883,2.125,medium,0.688,0.521,0.395
3,First Class,arizona,31,15.539,2.802,good,0.412,0.794,0.397
4,Second Class,arizona,24,16.020,1.956,medium,0.511,0.453,0.518
...,...,...,...,...,...,...,...,...,...
102,Second Class,washington,94,14.938,2.495,good,0.289,0.670,0.520
103,Same Day,washington,32,16.391,2.511,good,0.587,0.676,0.369
104,First Class,washington,56,16.222,2.601,good,0.552,0.713,0.367
105,First Class,wisconsin,12,18.378,3.172,medium,0.995,0.943,0.031


### Best `shipping method` across different `customer-states`.

In [7524]:
ship_state_grouped.loc[ship_state_grouped.groupby('state_customer')['efficiency_score'].idxmax()].reset_index(drop=True)

,ship_mode,state_customer,total_shipments,avg_lead_time,avg_lead_variability,confidence,norm_lead_time,norm_variability,efficiency_score
0,Standard Class,alabama,30,15.411,2.339,good,0.386,0.607,0.504
1,Second Class,alberta,12,16.883,2.125,medium,0.688,0.521,0.395
2,Standard Class,arizona,121,14.375,2.525,good,0.173,0.682,0.572
3,Standard Class,arkansas,32,15.110,2.871,good,0.324,0.822,0.427
4,Standard Class,british columbia,20,16.906,2.889,medium,0.693,0.829,0.239
5,Standard Class,california,986,13.532,2.529,good,0.000,0.684,0.658
6,Standard Class,colorado,72,14.865,2.068,good,0.274,0.498,0.614
7,Standard Class,connecticut,38,16.195,2.631,good,0.547,0.725,0.364
8,Second Class,delaware,13,14.740,0.832,medium,0.248,0.000,0.876
9,Standard Class,florida,197,14.328,2.443,good,0.163,0.649,0.594


## Efficiency Benchmarking

### Performance of `factories` → `customer-regions` routes.

In [7231]:
factory_region_grouped = preprocessed_df.groupby(['factory', 'region_customer']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('avg_lead_time', ascending=True).reset_index()

In [7233]:
factory_region_grouped.insert(loc=0, 
                   column='factory → region(customers)', 
                   value = factory_region_grouped['factory'] + " → " + factory_region_grouped['region_customer'])

In [7235]:
factory_region_grouped.drop(columns=['factory', 'region_customer'], inplace=True)
factory_region_grouped

,factory → region(customers),total_shipments,avg_lead_time,avg_lead_variability
0,sugar shack → gulf,4,11.000,2.309
1,the other factory → interior,11,12.091,2.700
2,the other factory → gulf,19,12.737,2.621
3,secret factory → interior,45,12.800,2.982
4,the other factory → atlantic,37,12.946,2.460
5,secret factory → pacific,63,13.048,2.630
6,secret factory → gulf,35,13.057,2.900
7,lots o nuts → gulf,745,13.097,2.525
8,the other factory → pacific,31,13.097,2.427
9,wicked choccys → gulf,547,13.139,2.489


In [7237]:
y = factory_region_grouped['total_shipments'].sort_values().tolist()
q(y, [0.1,0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9])

{'10%': [8, 18.0, 2.0],
 '15%': [11, 17.0, 3.0],
 '20%': [18, 16.0, 4.0],
 '25%': [19, 15.0, 5.0],
 '30%': [31, 14.0, 6.0],
 '35%': [35, 13.0, 7.0],
 '40%': [37, 12.0, 8.0],
 '45%': [45, 11.0, 9.0],
 '50%': [63, 10.0, 10.0],
 '60%': [547, 8.0, 12.0],
 '70%': [802, 6.0, 14.0],
 '80%': [1113, 4.0, 16.0],
 '90%': [1378, 2.0, 18.0]}

### Setting the `threshold value` = 8, which equals ~20 percentile in `total shipments` across `regional routes`.

In [7240]:
threshold = 18

In [7242]:
factory_region_grouped = factory_region_grouped[factory_region_grouped['total_shipments'] >= threshold]
factory_region_grouped

,factory → region(customers),total_shipments,avg_lead_time,avg_lead_variability
2,the other factory → gulf,19,12.737,2.621
3,secret factory → interior,45,12.800,2.982
4,the other factory → atlantic,37,12.946,2.460
5,secret factory → pacific,63,13.048,2.630
6,secret factory → gulf,35,13.057,2.900
7,lots o nuts → gulf,745,13.097,2.525
8,the other factory → pacific,31,13.097,2.427
9,wicked choccys → gulf,547,13.139,2.489
10,lots o nuts → pacific,1524,13.146,2.513
11,wicked choccys → atlantic,973,13.152,2.436


### Creating the `confidence` column.

In [7245]:
factory_region_grouped.insert(4, column="confidence", value=factory_region_grouped.apply(lambda x: 'good' if x['total_shipments']>=30 else \
                                                               'medium' if x['total_shipments'] >= 20 else \
                                                               'weak'
                                                               , axis=1))
factory_region_grouped

,factory → region(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence
2,the other factory → gulf,19,12.737,2.621,weak
3,secret factory → interior,45,12.800,2.982,good
4,the other factory → atlantic,37,12.946,2.460,good
5,secret factory → pacific,63,13.048,2.630,good
6,secret factory → gulf,35,13.057,2.900,good
7,lots o nuts → gulf,745,13.097,2.525,good
8,the other factory → pacific,31,13.097,2.427,good
9,wicked choccys → gulf,547,13.139,2.489,good
10,lots o nuts → pacific,1524,13.146,2.513,good
11,wicked choccys → atlantic,973,13.152,2.436,good


### Using the `upper_bound_CI` as `average lead time`.

In [7248]:
factory_region_grouped.loc[:, 'avg_lead_time'] = \
                                factory_region_grouped.apply(lambda x: \
                                                           round(z_upper_bound(x['avg_lead_time'], 
                                                            x['avg_lead_variability'], 
                                                            x['total_shipments'] ), 3) if x['confidence'] == 'good' else \
                                                               
                                                           round(t_upper_bound(x['avg_lead_time'],
                                                                        x['avg_lead_variability'],
                                                                        x['total_shipments']), 3) 
                                                         , axis = 1)

factory_region_grouped

,factory → region(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence
2,the other factory → gulf,19,14.000,2.621,weak
3,secret factory → interior,45,13.671,2.982,good
4,the other factory → atlantic,37,13.739,2.460,good
5,secret factory → pacific,63,13.697,2.630,good
6,secret factory → gulf,35,14.018,2.900,good
7,lots o nuts → gulf,745,13.278,2.525,good
8,the other factory → pacific,31,13.951,2.427,good
9,wicked choccys → gulf,547,13.348,2.489,good
10,lots o nuts → pacific,1524,13.272,2.513,good
11,wicked choccys → atlantic,973,13.305,2.436,good


### Efficiency score

In [7251]:
## normalized average lead time
factory_region_grouped.insert(5, column='norm_lead_time', value = 
            round((factory_region_grouped['avg_lead_time'] - factory_region_grouped['avg_lead_time'].min()) / \
            (factory_region_grouped['avg_lead_time'].max() - factory_region_grouped['avg_lead_time'].min()), 3))

## normalized average lead time variability
factory_region_grouped.insert(6, column='norm_variability', value = 
            round((factory_region_grouped['avg_lead_variability'] - factory_region_grouped['avg_lead_variability'].min()) / \
            (factory_region_grouped['avg_lead_variability'].max() - factory_region_grouped['avg_lead_variability'].min()),3 ))

## efficiency score
factory_region_grouped.insert(7, column='efficiency_score', value = 
                round(1 - ((0.5 * factory_region_grouped['norm_lead_time']) + (0.5 * factory_region_grouped['norm_variability'])), 3))

### Best performing `routes` across `customer-regions` based on their efficiency.

In [7256]:
factory_region_grouped.sort_values('efficiency_score', ascending=False)[:10].reset_index(drop=True)

,factory → region(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence,norm_lead_time,norm_variability,efficiency_score
0,wicked choccys → atlantic,973,13.305,2.436,good,0.020,0.020,0.980
1,lots o nuts → atlantic,1378,13.381,2.425,good,0.065,0.000,0.968
2,lots o nuts → interior,1113,13.392,2.439,good,0.072,0.025,0.952
3,lots o nuts → pacific,1524,13.272,2.513,good,0.000,0.158,0.921
4,wicked choccys → gulf,547,13.348,2.489,good,0.045,0.115,0.920
5,lots o nuts → gulf,745,13.278,2.525,good,0.004,0.180,0.908
6,wicked choccys → pacific,1123,13.426,2.496,good,0.092,0.127,0.890
7,wicked choccys → interior,802,13.392,2.524,good,0.072,0.178,0.875
8,the other factory → atlantic,37,13.739,2.460,good,0.279,0.063,0.829
9,the other factory → pacific,31,13.951,2.427,good,0.406,0.004,0.795


### Least efficient `routes` across `customer-regions` based on their efficiency.

In [7182]:
factory_region_grouped.sort_values('efficiency_score', ascending=True)[:10].reset_index(drop=True)

,factory → region(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence,norm_lead_time,norm_variability,efficiency_score
0,secret factory → interior,45,13.671,2.982,good,0.066,1.000,0.467
1,secret factory → gulf,35,14.018,2.900,good,0.124,0.934,0.471
2,secret factory → atlantic,70,14.102,2.631,good,0.138,0.719,0.572
3,secret factory → pacific,63,13.697,2.630,good,0.070,0.718,0.606
4,the other factory → atlantic,37,13.739,2.460,good,0.077,0.582,0.671
5,wicked choccys → interior,802,13.392,2.524,good,0.020,0.634,0.673
6,lots o nuts → gulf,745,13.278,2.525,good,0.001,0.634,0.682
7,wicked choccys → pacific,1123,13.426,2.496,good,0.026,0.611,0.682
8,lots o nuts → pacific,1524,13.272,2.513,good,0.000,0.625,0.688
9,wicked choccys → gulf,547,13.348,2.489,good,0.013,0.606,0.690


### Performance of `factories` → `customer-states` routes.

In [7258]:
factory_state_grouped = preprocessed_df.groupby(['factory', 'state_customer']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('avg_lead_time', ascending=True).reset_index()

In [7260]:
factory_state_grouped.insert(loc=0, 
                   column='factory → state(customers)', 
                   value = factory_state_grouped['factory'] + " → " + factory_state_grouped['state_customer'])

In [7262]:
factory_state_grouped.drop(columns=['factory', 'state_customer'], inplace=True)
factory_state_grouped

,factory → state(customers),total_shipments,avg_lead_time,avg_lead_variability
0,secret factory → minnesota,1,9.0,NaN
1,the other factory → louisiana,1,9.0,NaN
2,secret factory → delaware,1,9.0,NaN
3,wicked choccys → maine,2,9.0,0.0
4,secret factory → mississippi,1,9.0,NaN
...,...,...,...,...
191,secret factory → connecticut,1,16.0,NaN
192,wicked choccys → north dakota,1,16.0,NaN
193,lots o nuts → west virginia,2,16.0,0.0
194,wicked choccys → west virginia,2,16.0,0.0


### `Customer-states` that have only `1 shipment` for a specific route. 
- Better to filter them out as noise as we cannot rely on their average lead time.

In [7265]:
factory_state_grouped[factory_state_grouped['avg_lead_variability'].isna()]

,factory → state(customers),total_shipments,avg_lead_time,avg_lead_variability
0,secret factory → minnesota,1,9.0,NaN
1,the other factory → louisiana,1,9.0,NaN
2,secret factory → delaware,1,9.0,NaN
4,secret factory → mississippi,1,9.0,NaN
7,secret factory → nebraska,1,9.0,NaN
9,secret factory → south carolina,1,9.0,NaN
10,sugar shack → delaware,1,9.0,NaN
66,the other factory → indiana,1,13.0,NaN
67,sugar shack → tennessee,1,13.0,NaN
68,sugar shack → rhode island,1,13.0,NaN


In [7267]:
y = factory_state_grouped['total_shipments'].sort_values().tolist()

In [7269]:
q(y, [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9])

{'20%': [2, 156.0, 40.0],
 '25%': [2, 147.0, 49.0],
 '30%': [3, 137.0, 59.0],
 '35%': [3, 127.0, 69.0],
 '40%': [5, 117.0, 79.0],
 '45%': [6, 107.0, 89.0],
 '50%': [8, 98.0, 98.0],
 '60%': [16, 78.0, 118.0],
 '70%': [24, 58.0, 138.0],
 '80%': [44, 39.0, 157.0],
 '90%': [94, 19.0, 177.0]}

In [7273]:
factory_state_grouped['total_shipments'].quantile(.6)

16.0

### Setting the `threshold value` = 16, which equals ~20 percentile in `total shipments` across `state routes`.

In [7276]:
threshold = 16

In [7278]:
factory_state_grouped =  factory_state_grouped[factory_state_grouped['total_shipments'] >= threshold]
factory_state_grouped

,factory → state(customers),total_shipments,avg_lead_time,avg_lead_variability
23,lots o nuts → utah,21,12.000,2.898
25,secret factory → texas,16,12.188,3.124
27,lots o nuts → virginia,91,12.341,2.441
28,wicked choccys → mississippi,17,12.412,2.181
29,lots o nuts → ontario,24,12.458,2.553
...,...,...,...,...
147,lots o nuts → maryland,44,13.795,2.097
151,secret factory → new york,23,13.913,2.410
152,wicked choccys → tennessee,56,13.929,2.122
160,lots o nuts → connecticut,40,14.075,2.390


### Creating the `confidence` column.

In [6863]:
factory_state_grouped.insert(4, column='confidence', value= factory_state_grouped.apply(lambda x: 'good' if x['total_shipments']>=30 else \
                                                               'medium' if x['total_shipments'] >= 10 else \
                                                               'weak'
                                                               , axis=1))
factory_state_grouped

,factory → state(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence
23,lots o nuts → utah,21,12.000,2.898,medium
25,secret factory → texas,16,12.188,3.124,medium
27,lots o nuts → virginia,91,12.341,2.441,good
28,wicked choccys → mississippi,17,12.412,2.181,medium
29,lots o nuts → ontario,24,12.458,2.553,medium
...,...,...,...,...,...
147,lots o nuts → maryland,44,13.795,2.097,good
151,secret factory → new york,23,13.913,2.410,medium
152,wicked choccys → tennessee,56,13.929,2.122,good
160,lots o nuts → connecticut,40,14.075,2.390,good


### Using the `upper_bound_CI` as `average lead time`.

In [6866]:
factory_state_grouped.loc[:, 'avg_lead_time'] = \
                                factory_state_grouped.apply(lambda x: \
                                                            round(z_upper_bound(x['avg_lead_time'], 
                                                            x['avg_lead_variability'], 
                                                            x['total_shipments'] ), 3) if x['confidence'] == 'good' else \
                                                               
                                                           round(t_upper_bound(x['avg_lead_time'],
                                                                        x['avg_lead_variability'],
                                                                        x['total_shipments']), 3) 
                                                         , axis = 1)

factory_state_grouped.reset_index(drop=True, inplace=True)
factory_state_grouped

,factory → state(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence
0,lots o nuts → utah,21,13.319,2.898,medium
1,secret factory → texas,16,13.853,3.124,medium
2,lots o nuts → virginia,91,12.843,2.441,good
3,wicked choccys → mississippi,17,13.533,2.181,medium
4,lots o nuts → ontario,24,13.536,2.553,medium
...,...,...,...,...,...
74,lots o nuts → maryland,44,14.415,2.097,good
75,secret factory → new york,23,14.955,2.410,medium
76,wicked choccys → tennessee,56,14.485,2.122,good
77,lots o nuts → connecticut,40,14.816,2.390,good


## Efficiency score

In [6870]:
## normalized average lead time
factory_state_grouped.insert(5, column='norm_lead_time', value = 
            round((factory_state_grouped['avg_lead_time'] - factory_state_grouped['avg_lead_time'].min()) / \
            (factory_state_grouped['avg_lead_time'].max() - factory_state_grouped['avg_lead_time'].min()), 3))

## normalized average lead time variability
factory_state_grouped.insert(6, column='norm_variability', value = 
            round((factory_state_grouped['avg_lead_variability'] - factory_state_grouped['avg_lead_variability'].min()) / \
            (factory_state_grouped['avg_lead_variability'].max() - factory_state_grouped['avg_lead_variability'].min()),3 ))

## efficiency score
factory_state_grouped.insert(7, column='efficiency_score', value = 
                round(1 - ((0.5 * factory_state_grouped['norm_lead_time']) + (0.5 * factory_state_grouped['norm_variability'])), 3))

### Best performing `routes` across `customer-states`.

In [6893]:
factory_state_grouped.sort_values('efficiency_score', ascending=False)[:10].reset_index(drop=True)

,factory → state(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence,norm_lead_time,norm_variability,efficiency_score
0,wicked choccys → delaware,37,13.596,2.100,good,0.276,0.071,0.826
1,wicked choccys → wisconsin,42,13.850,2.022,good,0.369,0.000,0.816
2,lots o nuts → virginia,91,12.843,2.441,good,0.000,0.380,0.810
3,wicked choccys → mississippi,17,13.533,2.181,medium,0.253,0.144,0.802
4,wicked choccys → new york,353,13.338,2.350,good,0.181,0.298,0.760
5,wicked choccys → new jersey,49,13.960,2.118,good,0.409,0.087,0.752
6,wicked choccys → virginia,85,13.038,2.501,good,0.071,0.435,0.747
7,lots o nuts → new york,544,13.487,2.341,good,0.236,0.289,0.738
8,wicked choccys → michigan,83,13.569,2.310,good,0.266,0.261,0.736
9,lots o nuts → oregon,62,13.150,2.481,good,0.112,0.417,0.736


### Least efficient `routes` across `customer-states`.

In [6896]:
factory_state_grouped.sort_values('efficiency_score', ascending=True)[:10].reset_index(drop=True)

,factory → state(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence,norm_lead_time,norm_variability,efficiency_score
0,wicked choccys → rhode island,18,15.029,2.852,medium,0.801,0.753,0.223
1,wicked choccys → quebec,16,14.466,2.986,medium,0.595,0.875,0.265
2,secret factory → texas,16,13.853,3.124,medium,0.370,1.000,0.315
3,lots o nuts → quebec,20,14.520,2.821,medium,0.614,0.725,0.330
4,wicked choccys → arkansas,21,14.319,2.898,medium,0.541,0.795,0.332
5,lots o nuts → new mexico,18,15.573,2.380,medium,1.000,0.325,0.338
6,wicked choccys → minnesota,36,14.182,2.939,good,0.490,0.832,0.339
7,lots o nuts → british columbia,18,14.134,2.951,medium,0.473,0.843,0.342
8,secret factory → pennsylvania,19,14.411,2.818,medium,0.574,0.722,0.352
9,lots o nuts → arkansas,25,14.130,2.835,medium,0.471,0.738,0.395


## Geographic Bottleneck Analysis

### `Regions` with `High` average lead time

In [6970]:
least_efficient_routes =  factory_region_grouped.sort_values('efficiency_score', ascending=True)[:6].reset_index(drop=True)
least_efficient_routes

,factory → region(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence,norm_lead_time,norm_variability,efficiency_score
0,secret factory → gulf,35,14.018,2.899725,good,0.446,0.853,0.351
1,sugar shack → atlantic,18,14.943,2.566756,weak,1.000,0.255,0.373
2,secret factory → interior,45,13.671,2.981763,good,0.239,1.000,0.381
3,secret factory → atlantic,70,14.102,2.630605,good,0.497,0.370,0.566
4,the other factory → gulf,19,14.000,2.621325,weak,0.436,0.353,0.606
5,secret factory → pacific,63,13.697,2.630029,good,0.254,0.369,0.688


### Analyzing the `shipment volume` for the `least` efficient `routes` across `regions`.

In [6973]:
y = least_efficient_routes['total_shipments'].tolist()
y.sort()

In [6978]:
q(y, [0.05, 0.1,0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8])

{'5%': [19, 5.0, 1.0],
 '10%': [19, 5.0, 1.0],
 '15%': [19, 5.0, 1.0],
 '20%': [35, 4.0, 2.0],
 '25%': [35, 4.0, 2.0],
 '30%': [35, 4.0, 2.0],
 '35%': [45, 3.0, 3.0],
 '40%': [45, 3.0, 3.0],
 '45%': [45, 3.0, 3.0],
 '50%': [45, 3.0, 3.0],
 '60%': [63, 2.0, 4.0],
 '70%': [70, 1.0, 5.0],
 '80%': [70, 1.0, 5.0]}

### Setting the `threshold value` = 35, which equals ~30 percentile in `total shipments` across `regional routes`.

In [6981]:
threshold = 35

In [6988]:
bottleneck_regions = least_efficient_routes[least_efficient_routes['total_shipments'] >= threshold].reset_index(drop=True)

### Identifying the `regional routes` with:
1. high `average lead time`
2. high `shipment volume`
3. consistently `poor performance`

In [6991]:
bottleneck_regions

,factory → region(customers),total_shipments,avg_lead_time,avg_lead_variability,confidence,norm_lead_time,norm_variability,efficiency_score
0,secret factory → gulf,35,14.018,2.899725,good,0.446,0.853,0.351
1,secret factory → interior,45,13.671,2.981763,good,0.239,1.000,0.381
2,secret factory → atlantic,70,14.102,2.630605,good,0.497,0.370,0.566
3,secret factory → pacific,63,13.697,2.630029,good,0.254,0.369,0.688


### Identifying the `congestion` prone `regions`

In [7023]:
congested_prone_regions = preprocessed_df.groupby(['region_customer']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('avg_lead_time', ascending=False).reset_index()
congested_prone_regions 

,region_customer,total_shipments,avg_lead_time,avg_lead_variability
0,interior,1979,13.219,2.487
1,atlantic,2476,13.218,2.436
2,pacific,2744,13.200,2.507
3,gulf,1350,13.101,2.521


In [7025]:
## normalized average lead time
congested_prone_regions.insert(4, column='norm_lead_time', value = 
            round((congested_prone_regions['avg_lead_time'] - congested_prone_regions['avg_lead_time'].min()) / \
            (congested_prone_regions['avg_lead_time'].max() - congested_prone_regions['avg_lead_time'].min()), 3))

## normalized average lead time variability
congested_prone_regions.insert(5, column='norm_variability', value = 
            round((congested_prone_regions['avg_lead_variability'] - congested_prone_regions['avg_lead_variability'].min()) / \
            (congested_prone_regions['avg_lead_variability'].max() - congested_prone_regions['avg_lead_variability'].min()),3 ))

## efficiency score
congested_prone_regions.insert(6, column='efficiency_score', value = 
                round(1 - ((0.5 * congested_prone_regions['norm_lead_time']) + (0.5 * congested_prone_regions['norm_variability'])), 3))

In [7027]:
congested_prone_regions.sort_values('efficiency_score', ascending=True).reset_index(drop=True).style.apply(
                    lambda x:  ['background-color: lightblue']*len(x) if x['efficiency_score'] <= 0.2 else ['']*len(x),
                    axis=1
                            )

,region_customer,total_shipments,avg_lead_time,avg_lead_variability,norm_lead_time,norm_variability,efficiency_score
0,pacific,2744,13.200000,2.507000,0.839000,0.835000,0.163000
1,interior,1979,13.219000,2.487000,1.000000,0.600000,0.200000
2,gulf,1350,13.101000,2.521000,0.000000,1.000000,0.500000
3,atlantic,2476,13.218000,2.436000,0.992000,0.000000,0.504000


### Identifying the congestion prone states

In [7061]:
grouped_states = preprocessed_df.groupby(['state_customer']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('avg_lead_time', ascending=False).reset_index()

In [7063]:
y = grouped_states['total_shipments'].sort_values().tolist()

In [7065]:
q(y, [0.05, 0.1,0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])

{'5%': [5, 56.0, 3.0],
 '10%': [7, 53.0, 6.0],
 '15%': [8, 50.0, 9.0],
 '20%': [10, 47.0, 12.0],
 '25%': [22, 44.0, 15.0],
 '30%': [26, 41.0, 18.0],
 '35%': [34, 38.0, 21.0],
 '40%': [38, 35.0, 24.0],
 '45%': [43, 32.0, 27.0],
 '50%': [48, 29.0, 30.0],
 '60%': [80, 23.0, 36.0],
 '70%': [112, 17.0, 42.0],
 '80%': [190, 11.0, 48.0],
 '90%': [433, 5.0, 54.0],
 '95%': [942, 2.0, 57.0]}

### Setting the `threshold value` = 34, which equals ~35 percentile in `total shipments` across `states`.

In [7068]:
threshold = 34 # ~35%

In [7070]:
congested_prone_states = grouped_states[grouped_states['total_shipments'] >= threshold]

### Efficiency score

In [7073]:
## normalized average lead time
congested_prone_states.insert(4, column='norm_lead_time', value = 
            round((congested_prone_states['avg_lead_time'] - congested_prone_states['avg_lead_time'].min()) / \
            (congested_prone_states['avg_lead_time'].max() - congested_prone_states['avg_lead_time'].min()), 3))

## normalized average lead time variability
congested_prone_states.insert(5, column='norm_variability', value = 
            round((congested_prone_states['avg_lead_variability'] - congested_prone_states['avg_lead_variability'].min()) / \
            (congested_prone_states['avg_lead_variability'].max() - congested_prone_states['avg_lead_variability'].min()),3 ))

## efficiency score
congested_prone_states.insert(6, column='efficiency_score', value = 
                round(1 - ((0.5 * congested_prone_states['norm_lead_time']) + (0.5 * congested_prone_states['norm_variability'])), 3))

In [7075]:
congested_prone_states.sort_values('efficiency_score', ascending=True).reset_index(drop=True).style.apply(
                                lambda x:  ['background-color: salmon']*len(x) if x['efficiency_score'] <= 0.5 else ['']*len(x),
                                axis=1
                            )

,state_customer,total_shipments,avg_lead_time,avg_lead_variability,norm_lead_time,norm_variability,efficiency_score
0,new mexico,34,14.088000,2.527000,1.000000,0.430000,0.285000
1,minnesota,75,13.267000,2.816000,0.524000,0.777000,0.349000
2,quebec,38,13.026000,2.899000,0.385000,0.876000,0.369000
3,connecticut,74,13.446000,2.628000,0.628000,0.552000,0.410000
4,arkansas,48,13.042000,2.805000,0.394000,0.764000,0.421000
5,rhode island,46,13.457000,2.605000,0.634000,0.524000,0.421000
6,washington,432,13.572000,2.529000,0.701000,0.433000,0.433000
7,louisiana,38,12.553000,3.002000,0.111000,1.000000,0.444000
8,kentucky,112,13.304000,2.574000,0.546000,0.487000,0.484000
9,missouri,56,13.339000,2.553000,0.566000,0.462000,0.486000


### `Expedited Shipping` Vs `Standard Shipping`

In [7108]:
shipping =  preprocessed_df.groupby(['ship_mode']).agg(
    total_shipments = ('Order ID', 'count'),
    avg_lead_time = ('lead_time', lambda x: round(x.mean(), 3)),
    avg_lead_variability = ('lead_time', lambda x: round(x.std(), 3)),
).sort_values('avg_lead_time', ascending=False).reset_index()

In [7110]:
shipping['shipping_category'] = shipping_efficiency['ship_mode']\
                .apply(lambda x: 'Expedited Shipping' if x in ('First Class', 'Same Day') else 'Standard Shipping')
                                        

In [7121]:
shipping

,ship_mode,total_shipments,avg_lead_time,avg_lead_variability,shipping_category
0,First Class,1312,13.349,2.503,Expedited Shipping
1,Same Day,442,13.303,2.416,Expedited Shipping
2,Second Class,1653,13.196,2.468,Standard Shipping
3,Standard Class,5142,13.145,2.489,Standard Shipping


In [7138]:
shipping_grouped = shipping_efficiency.groupby('shipping_category').agg(
            total_shipments = ('total_shipments', 'sum'),
            avg_lead_time = ('avg_lead_time', lambda x: round(x.mean(), 3)),
            avg_lead_variability = ('avg_lead_variability', lambda x: round(x.mean(), 3)),
        ).sort_values('avg_lead_time', ascending=False).reset_index()

In [7140]:
shipping_grouped

,shipping_category,total_shipments,avg_lead_time,avg_lead_variability
0,Expedited Shipping,1754,13.326,2.460
1,Standard Shipping,6795,13.170,2.478


### Efficiency score

In [7143]:
## normalized average lead time
shipping_grouped.insert(4, column='norm_lead_time', value = 
            round((shipping_grouped['avg_lead_time'] - shipping_grouped['avg_lead_time'].min()) / \
            (shipping_grouped['avg_lead_time'].max() - shipping_grouped['avg_lead_time'].min()), 4))

## normalized average lead time variability
shipping_grouped.insert(5, column='norm_variability', value = 
            round((shipping_grouped['avg_lead_variability'] - shipping_grouped['avg_lead_variability'].min()) / \
            (shipping_grouped['avg_lead_variability'].max() - shipping_grouped['avg_lead_variability'].min()),4 ))

## efficiency score
shipping_grouped.insert(6, column='efficiency_score', value = 
                round(1 - ((0.5 * shipping_grouped['norm_lead_time']) + (0.5 * shipping_grouped['norm_variability'])), 4))

In [7145]:
shipping_grouped

,shipping_category,total_shipments,avg_lead_time,avg_lead_variability,norm_lead_time,norm_variability,efficiency_score
0,Expedited Shipping,1754,13.326,2.460,1.0,0.0,0.5
1,Standard Shipping,6795,13.170,2.478,0.0,1.0,0.5
